![TecNM](assets/encabezado.png)
---

# Machine Learning y Deep Learning
## Unidad 2 · Modelos de Predicción Supervisados

Práctica 1 — Regresión Lineal Simple

> **Facilitador:** Dr. José Gabriel Rodríguez Rivas  
> **Alumno:** Christian Gibran Espituñal Villanueva

---

## Objetivo

Construir e interpretar un modelo de **Regresión Lineal Simple** para predecir el precio de un vehículo a partir de su consumo urbano (`city-mpg`), evaluando el rendimiento con métricas estándar y analizando las limitaciones del modelo de una sola variable.

---

## Marco Teórico

La **Regresión Lineal Simple** modela la relación entre una variable predictora $x$ y una variable respuesta $y$ mediante la recta:

$$\hat{y} = \beta_0 + \beta_1 x$$

donde $\beta_0$ es el intercepto (bias) y $\beta_1$ es el coeficiente de pendiente. El ajuste se obtiene minimizando el error cuadrático medio (MSE) sobre el conjunto de entrenamiento.

Cuando se dispone de GPU, los cálculos de álgebra lineal pueden ejecutarse en CUDA utilizando `torch.linalg.lstsq`, que resuelve el sistema de ecuaciones normales $\mathbf{X}^T\mathbf{X}\boldsymbol{\beta} = \mathbf{X}^T\mathbf{y}$ directamente en la GPU.

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Error promedio al cuadrado |
| **RMSE** | $\sqrt{\text{MSE}}$ | Error en las mismas unidades que $y$ |
| **MAE** | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Error absoluto promedio |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporción de varianza explicada |

---

## Conjunto de Datos

El archivo `autos2.csv` contiene especificaciones técnicas y precios de mercado de automóviles. En esta práctica se emplean:

- **Variable predictora:** `city-mpg` — Consumo urbano (millas por galón)
- **Variable objetivo:** `price` — Precio del vehículo (USD)

---

## Contenido

1. [Entorno y librerías](#1.-Entorno-y-librerías)
2. [Detección y configuración de dispositivo (CPU / CUDA)](#2.-Detección-y-configuración-de-dispositivo)
3. [Carga y exploración inicial](#3.-Carga-y-exploración-inicial)
4. [Análisis exploratorio](#4.-Análisis-exploratorio)
5. [Preprocesamiento y división](#5.-Preprocesamiento-y-división)
6. [Entrenamiento del modelo](#6.-Entrenamiento-del-modelo)
7. [Evaluación de métricas](#7.-Evaluación-de-métricas)
8. [Visualización de resultados](#8.-Visualización-de-resultados)
9. [Diagnóstico de residuos](#9.-Diagnóstico-de-residuos)
10. [Interpretación y conclusiones](#10.-Interpretación-y-conclusiones)

---
## 1. Entorno y librerías

In [ ]:
# ── Librerías estándar ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
import scipy.stats as stats
import time

# ── PyTorch (detección CUDA + regresión en GPU) ───────────────────────────────
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cuda.is_built())

# ── cuML — regresión acelerada por GPU con RAPIDS (opcional) ──────────────────
try:
    from cuml.linear_model import LinearRegression as cuLinearRegression
    CUML_AVAILABLE = True
except ImportError:
    CUML_AVAILABLE = False

# ── Scikit-learn (CPU / fallback universal) ───────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Estilo global de visualizaciones ─────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'font.family': 'DejaVu Sans',
})

C_BLUE   = '#005F9E'
C_ORANGE = '#E87722'
C_GREEN  = '#3DAD6B'
C_RED    = '#C0392B'
C_PURPLE = '#8E44AD'
PALETTE  = [C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE]

sns.set_theme(style='whitegrid', palette=PALETTE)

RANDOM_STATE = 42
FEATURE      = 'city-mpg'
TARGET       = 'price'

print('✅ Entorno configurado correctamente.')

2.10.0+cpu
None
False
✅ Entorno configurado correctamente.


---
## 2. Detección y configuración de dispositivo

El notebook detecta automáticamente el hardware disponible y selecciona el backend óptimo:

| Escenario | Biblioteca de regresión | Backend de álgebra lineal |
|---|---|---|
| GPU + cuML (RAPIDS) | `cuml.LinearRegression` | CUDA nativo |
| GPU sin cuML | `torch.linalg.lstsq` | CUDA vía PyTorch |
| Sin GPU | `sklearn.LinearRegression` | CPU (NumPy / LAPACK) |

In [ ]:
# ── Detección del dispositivo ─────────────────────────────────────────────────
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE         = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

print('╔══════════════════════════════════════════════╗')
print('║         Información del dispositivo          ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  CUDA disponible : {str(CUDA_AVAILABLE):<27}║')
print(f'║  Dispositivo     : {str(DEVICE):<27}║')

if CUDA_AVAILABLE:
    gpu_name   = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    vram_free  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1024**3
    cuda_ver   = torch.version.cuda
    sm_major   = torch.cuda.get_device_properties(0).major
    sm_minor   = torch.cuda.get_device_properties(0).minor
    n_sms      = torch.cuda.get_device_properties(0).multi_processor_count
    print(f'║  GPU             : {gpu_name[:27]:<27}║')
    print(f'║  Compute cap.    : {f"sm_{sm_major}{sm_minor} ({n_sms} SMs)":<27}║')
    print(f'║  VRAM total      : {f"{vram_total:.2f} GB":<27}║')
    print(f'║  VRAM libre      : {f"{vram_free:.2f} GB":<27}║')
    print(f'║  CUDA versión    : {cuda_ver:<27}║')
    print(f'║  PyTorch versión : {torch.__version__:<27}║')
    print(f'║  cuML disponible : {str(CUML_AVAILABLE):<27}║')
    if CUML_AVAILABLE:
        print('║  ✅ Backend      : cuML (RAPIDS) CUDA          ║')
    else:
        print('║  ✅ Backend      : PyTorch CUDA (lstsq)        ║')
else:
    print(f'║  PyTorch versión : {torch.__version__:<27}║')
    print('║  ✅ Backend      : scikit-learn CPU (LAPACK)   ║')
    print('║                                                ║')
    print('║  💡 Para habilitar GPU instala:                ║')
    print('║  pip install torch --index-url                 ║')
    print('║  https://download.pytorch.org/whl/cu121       ║')
    print('║  pip install cuml-cu12 (RAPIDS, opcional)      ║')
print('╚══════════════════════════════════════════════╝')

In [ ]:
# ── Clase unificada: misma API sin importar el backend ────────────────────────
class LinearRegressionDevice:
    """
    Wrapper de Regresión Lineal con soporte automático para:
      - CPU      → sklearn LinearRegression  (LAPACK/NumPy)
      - GPU      → torch.linalg.lstsq        (CUDA, sin dependencias extras)
      - GPU+cuML → cuml.LinearRegression     (RAPIDS, máxima performance)

    Expone la misma interfaz: .fit(X, y)  .predict(X)  .coef_  .intercept_
    """

    def __init__(self, device: torch.device, use_cuml: bool = False):
        self.device      = device
        self.use_cuml    = use_cuml and CUML_AVAILABLE and device.type == 'cuda'
        self.coef_       = None
        self.intercept_  = None

    # ── fit ───────────────────────────────────────────────────────────────────
    def fit(self, X: np.ndarray, y: np.ndarray):
        if self.use_cuml:
            self._fit_cuml(X, y)
        elif self.device.type == 'cuda':
            self._fit_torch_cuda(X, y)
        else:
            self._fit_sklearn(X, y)
        return self

    def _fit_sklearn(self, X, y):
        m = LinearRegression().fit(X, y)
        self.coef_      = m.coef_
        self.intercept_ = float(m.intercept_)

    def _fit_torch_cuda(self, X, y):
        """OLS exacto via mínimos cuadrados en CUDA: β = lstsq(X_aug, y)"""
        ones  = np.ones((X.shape[0], 1), dtype=np.float32)
        X_aug = np.hstack([ones, X.astype(np.float32)])
        y_col = y.astype(np.float32).reshape(-1, 1)

        X_t = torch.tensor(X_aug, dtype=torch.float32, device=self.device)
        y_t = torch.tensor(y_col, dtype=torch.float32, device=self.device)

        # driver='gels' usa QR-factorization (rápido y estable)
        result = torch.linalg.lstsq(X_t, y_t, driver='gels')
        betas  = result.solution.squeeze().cpu().numpy()

        self.intercept_ = float(betas[0])
        self.coef_      = betas[1:]

    def _fit_cuml(self, X, y):
        """Regresión con cuML (RAPIDS) — pipeline 100% en GPU."""
        import cupy as cp
        X_g = cp.asarray(X.astype(np.float32))
        y_g = cp.asarray(y.astype(np.float32))
        m   = cuLinearRegression().fit(X_g, y_g)
        self.coef_      = cp.asnumpy(m.coef_).flatten()
        self.intercept_ = float(cp.asnumpy(m.intercept_))

    # ── predict ───────────────────────────────────────────────────────────────
    def predict(self, X: np.ndarray) -> np.ndarray:
        return (X @ self.coef_ + self.intercept_).flatten()

    def __repr__(self):
        b = ('cuML-CUDA'     if self.use_cuml else
             'PyTorch-CUDA'  if self.device.type == 'cuda' else
             'sklearn-CPU')
        return f'LinearRegressionDevice(backend={b})'


backend_activo = ('cuML-CUDA'    if (CUML_AVAILABLE and CUDA_AVAILABLE) else
                  'PyTorch-CUDA' if CUDA_AVAILABLE else
                  'sklearn-CPU')
print(f'✅ LinearRegressionDevice lista → backend activo: {backend_activo}')

---
## 3. Carga y exploración inicial

In [ ]:
df = pd.read_csv('autos2.csv')

print(f'Dimensiones     : {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Valores nulos   : {df[[FEATURE, TARGET]].isnull().sum().to_dict()}')
print()
display(
    df[[FEATURE, TARGET]]
    .describe().T
    .style
    .format(precision=2)
    .background_gradient(cmap='Blues', subset=['mean', 'std'])
    .set_caption('Estadísticas descriptivas — variables de interés')
)

---
## 4. Análisis exploratorio

In [ ]:
data_clean = df[[FEATURE, TARGET]].dropna().reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Análisis exploratorio de variables', fontsize=14,
             fontweight='bold', y=1.02)

# ── Distribución de city-mpg ──────────────────────────────────────────────────
sns.histplot(data_clean[FEATURE], bins=25, kde=True,
             color=C_BLUE, edgecolor='white', ax=axes[0])
axes[0].set_title('Distribución de city-mpg')
axes[0].set_xlabel('Consumo urbano (mpg)')
axes[0].set_ylabel('Frecuencia')

# ── Distribución de price ─────────────────────────────────────────────────────
sns.histplot(data_clean[TARGET], bins=25, kde=True,
             color=C_ORANGE, edgecolor='white', ax=axes[1])
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[1].set_title('Distribución del precio')
axes[1].set_xlabel('Precio (USD)')
axes[1].set_ylabel('Frecuencia')

# ── Dispersión con IC 95% ─────────────────────────────────────────────────────
sns.regplot(
    x=FEATURE, y=TARGET, data=data_clean,
    scatter_kws={'alpha': 0.6, 'color': C_BLUE, 'edgecolors': 'white', 's': 55},
    line_kws={'color': C_RED, 'linewidth': 2.2},
    ci=95, ax=axes[2]
)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[2].set_title('city-mpg vs. price (con IC 95%)')
axes[2].set_xlabel('Consumo urbano (mpg)')
axes[2].set_ylabel('Precio (USD)')

r, p = stats.pearsonr(data_clean[FEATURE], data_clean[TARGET])
axes[2].text(0.05, 0.92, f'r = {r:.3f}  (p={p:.1e})',
             transform=axes[2].transAxes, fontsize=9, color='#333')

plt.tight_layout()
plt.show()

print(f'Correlación de Pearson: r = {r:.4f}  →  '
      f'{"relación negativa moderada" if r < -0.4 else "relación débil"}')

---
## 5. Preprocesamiento y división

In [ ]:
# Usar arrays numpy para compatibilidad con todos los backends
X = data_clean[[FEATURE]].values
y = data_clean[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Total de registros : {len(data_clean)}')
print(f'Entrenamiento      : {len(X_train)} muestras  ({len(X_train)/len(data_clean):.0%})')
print(f'Prueba             : {len(X_test)}  muestras  ({len(X_test)/len(data_clean):.0%})')

---
## 6. Entrenamiento del modelo

Se entrena el modelo con el backend detectado y se mide el tiempo de ajuste. Cuando CUDA está disponible, se ejecuta además un **benchmark CPU vs GPU** con 50 repeticiones.

In [ ]:
# ── Entrenamiento principal ───────────────────────────────────────────────────
model = LinearRegressionDevice(device=DEVICE, use_cuml=CUML_AVAILABLE)

if CUDA_AVAILABLE:
    torch.cuda.synchronize()
t0 = time.perf_counter()
model.fit(X_train, y_train)
if CUDA_AVAILABLE:
    torch.cuda.synchronize()
t_fit_ms = (time.perf_counter() - t0) * 1e3

beta0 = model.intercept_
beta1 = model.coef_[0]

print('═' * 55)
print(f'  Modelo : {model}')
print('═' * 55)
print(f'  price = {beta0:,.2f} + ({beta1:,.2f}) × city-mpg')
print()
print(f'  β₀ (Intercepto) : {beta0:>12,.2f} USD')
print(f'  β₁ (Pendiente)  : {beta1:>12,.2f} USD/mpg')
print(f'  Tiempo de ajuste: {t_fit_ms:>11.3f} ms  [{"GPU" if CUDA_AVAILABLE else "CPU"}]')
print('═' * 55)
print(f'\nCada mpg adicional reduce el precio en ~${abs(beta1):,.0f} USD.')

In [ ]:
# ── Benchmark CPU vs GPU (solo si CUDA está disponible) ───────────────────────
if CUDA_AVAILABLE:
    N_REPS = 50

    # Tiempos CPU (sklearn)
    times_cpu = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        LinearRegression().fit(X_train, y_train)
        times_cpu.append((time.perf_counter() - t0) * 1e3)

    # Tiempos GPU (PyTorch CUDA)
    _gpu_model = LinearRegressionDevice(device=DEVICE, use_cuml=False)
    times_gpu = []
    for _ in range(N_REPS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _gpu_model.fit(X_train, y_train)
        torch.cuda.synchronize()
        times_gpu.append((time.perf_counter() - t0) * 1e3)

    avg_cpu = np.mean(times_cpu)
    avg_gpu = np.mean(times_gpu)
    speedup  = avg_cpu / avg_gpu

    # Visualización del benchmark
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Benchmark — Tiempo de ajuste: CPU vs GPU ({N_REPS} repeticiones)',
                 fontsize=13, fontweight='bold')

    # Serie temporal
    axes[0].plot(times_cpu, color=C_BLUE,   lw=1.5, alpha=0.8,
                 label=f'CPU sklearn  (μ={avg_cpu:.2f} ms)')
    axes[0].plot(times_gpu, color=C_ORANGE, lw=1.5, alpha=0.8,
                 label=f'GPU PyTorch  (μ={avg_gpu:.2f} ms)')
    axes[0].axhline(avg_cpu, color=C_BLUE,   ls='--', lw=1, alpha=0.45)
    axes[0].axhline(avg_gpu, color=C_ORANGE, ls='--', lw=1, alpha=0.45)
    axes[0].set_xlabel('Iteración')
    axes[0].set_ylabel('Tiempo (ms)')
    axes[0].set_title('Serie temporal')
    axes[0].legend(frameon=True)

    # Box-plot comparativo
    bp_data = pd.DataFrame({'CPU (sklearn)': times_cpu, 'GPU (PyTorch)': times_gpu})
    bp_data.plot(kind='box', ax=axes[1], color={'boxes': C_BLUE, 'medians': C_RED,
                  'whiskers': C_BLUE, 'caps': C_BLUE},
                 boxprops=dict(linewidth=1.8),
                 medianprops=dict(linewidth=2.2, color=C_RED))
    axes[1].set_ylabel('Tiempo (ms)')
    axes[1].set_title(f'Distribución  (speedup GPU: ×{speedup:.1f})')

    plt.tight_layout()
    plt.show()

    print(f'  CPU promedio : {avg_cpu:.3f} ms')
    print(f'  GPU promedio : {avg_gpu:.3f} ms')
    print(f'  Speedup GPU  : ×{speedup:.1f}')
    print()
    print('Nota: para datasets pequeños el overhead de transferencia CPU↔GPU')
    print('puede superar la ganancia. El speedup real escala con N (filas).')
else:
    print('ℹ️  Benchmark omitido — CUDA no disponible.')
    print('   Conecta una GPU y vuelve a ejecutar para ver la comparación.')

---
## 7. Evaluación de métricas

In [ ]:
y_pred = model.predict(X_test)

mse   = mean_squared_error(y_test, y_pred)
rmse  = np.sqrt(mse)
mae   = mean_absolute_error(y_test, y_pred)
r2    = r2_score(y_test, y_pred)

# CV-R² usando sklearn en CPU (cross_val_score no soporta cuML directamente)
_sk   = LinearRegression().fit(X_train, y_train)
cv_r2 = cross_val_score(_sk, X_train, y_train, cv=5, scoring='r2').mean()

precio_medio = np.median(y)
rmse_pct     = rmse / precio_medio * 100

metricas = pd.DataFrame({
    'Métrica'        : ['MSE', 'RMSE', 'MAE', 'R² (Prueba)', 'CV-R² (5-fold)'],
    'Valor'          : [mse, rmse, mae, r2, cv_r2],
    'Unidad'         : ['USD²', 'USD', 'USD', '—', '—'],
    'Interpretación' : [
        'Error cuadrático promedio',
        f'Error típico ≈ {rmse_pct:.1f}% del precio mediano',
        'Error absoluto promedio',
        f'{r2:.0%} de la varianza explicada por el modelo',
        'Generalización estimada (validación cruzada 5-fold)'
    ]
})

display(
    metricas.style
    .format({'Valor': lambda v: f'{v:,.4f}' if abs(v) < 10 else f'{v:,.2f}'})
    .hide(axis='index')
    .set_caption('Métricas de evaluación del modelo')
    .applymap(lambda _: 'background-color: #D4EFDF',
              subset=pd.IndexSlice[[3, 4], 'Valor'])
)

---
## 8. Visualización de resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Evaluación visual — Regresión Lineal Simple (city-mpg → price)',
             fontsize=13, fontweight='bold', y=1.01)
fmt_usd = mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k')

# ── 1. KDE Real vs. Predicho ──────────────────────────────────────────────────
sns.kdeplot(y_test,  label='Real',     color=C_BLUE,   linewidth=2.2, ax=axes[0])
sns.kdeplot(y_pred,  label='Predicho', color=C_ORANGE,
            linewidth=2.2, linestyle='--', ax=axes[0])
axes[0].xaxis.set_major_formatter(fmt_usd)
axes[0].set_title('Distribución de densidad\nReal vs. Predicho')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Densidad')
axes[0].legend(frameon=True)
axes[0].annotate('Distribución bimodal\n(no capturada por el modelo)',
                 xy=(37000, 0.000012), fontsize=7.5, color='gray',
                 ha='center', style='italic')

# ── 2. Scatter real vs. predicho ──────────────────────────────────────────────
lim = (min(y_test.min(), y_pred.min()) * 0.9,
       max(y_test.max(), y_pred.max()) * 1.08)
axes[1].scatter(y_test, y_pred, color=C_BLUE, alpha=0.7,
                edgecolors='white', linewidths=0.4, s=60)
axes[1].plot(lim, lim, '--', color=C_RED, linewidth=1.8, label='Predicción perfecta')
axes[1].xaxis.set_major_formatter(fmt_usd)
axes[1].yaxis.set_major_formatter(fmt_usd)
axes[1].set_xlim(*lim)
axes[1].set_ylim(*lim)
axes[1].set_title('Real vs. Predicho')
axes[1].set_xlabel('Precio real (USD)')
axes[1].set_ylabel('Precio predicho (USD)')
axes[1].legend(fontsize=9, frameon=True)
axes[1].text(0.05, 0.92, f'R² = {r2:.3f}',
             transform=axes[1].transAxes, fontsize=10, color='#333')
axes[1].fill_between(lim, lim, [lim[0], lim[1]*1.5],
                     alpha=0.05, color=C_GREEN)
axes[1].fill_between(lim, [lim[0], lim[1]*1.5], lim,
                     alpha=0.05, color=C_RED)

# ── 3. Recta de regresión sobre los datos ─────────────────────────────────────
x_rng = np.linspace(X_train[:, 0].min(), X_train[:, 0].max(), 200).reshape(-1, 1)
y_rng = model.predict(x_rng)

axes[2].scatter(X_train[:, 0], y_train, color=C_BLUE, alpha=0.5,
                edgecolors='white', s=45, label='Entrenamiento')
axes[2].scatter(X_test[:, 0],  y_test,  color=C_ORANGE, alpha=0.8,
                edgecolors='white', s=55, marker='D', label='Prueba')
axes[2].plot(x_rng[:, 0], y_rng, color=C_RED, linewidth=2.2, label='Recta ajustada')
axes[2].yaxis.set_major_formatter(fmt_usd)
axes[2].set_title('Recta de regresión')
axes[2].set_xlabel('Consumo urbano (mpg)')
axes[2].set_ylabel('Precio (USD)')
axes[2].legend(fontsize=9, frameon=True)

plt.tight_layout()
plt.show()

---
## 9. Diagnóstico de residuos

Un buen modelo de regresión debe producir residuos distribuidos normalmente, con media cero y varianza constante (homocedasticidad).

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Panel de diagnóstico de residuos', fontsize=13,
             fontweight='bold', y=1.01)

# ── 1. Residuos vs. Fitted + LOWESS ──────────────────────────────────────────
axes[0].scatter(y_pred, residuos, color=C_BLUE, alpha=0.7,
                edgecolors='white', linewidths=0.4, s=60)
axes[0].axhline(0, color=C_RED, linestyle='--', linewidth=1.8)
from statsmodels.nonparametric.smoothers_lowess import lowess
smooth = lowess(residuos, y_pred, frac=0.6)
axes[0].plot(smooth[:, 0], smooth[:, 1], color=C_ORANGE, linewidth=2)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[0].set_title('Residuos vs. Valores predichos')
axes[0].set_xlabel('Valor predicho (USD)')
axes[0].set_ylabel('Residuo (USD)')

# ── 2. Histograma + KDE ───────────────────────────────────────────────────────
sns.histplot(residuos, bins=20, kde=True, color=C_GREEN,
             edgecolor='white', ax=axes[1])
axes[1].axvline(0, color=C_RED, linestyle='--', linewidth=1.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo (USD)')
axes[1].set_ylabel('Frecuencia')
axes[1].text(0.98, 0.95,
             f'Media: ${residuos.mean():,.0f}\nDesv.: ${residuos.std():,.0f}',
             transform=axes[1].transAxes, fontsize=8.5, va='top', ha='right',
             color='#333')

# ── 3. Q-Q plot ───────────────────────────────────────────────────────────────
(osm, osr), (slope, intercept_qq, r_qq) = stats.probplot(residuos, dist='norm')
axes[2].scatter(osm, osr, color=C_BLUE, alpha=0.7, s=55,
                edgecolors='white', linewidths=0.4)
x_ref = np.array([osm.min(), osm.max()])
axes[2].plot(x_ref, slope * x_ref + intercept_qq, color=C_RED,
             linewidth=2, linestyle='--')
axes[2].set_title('Q-Q plot de residuos')
axes[2].set_xlabel('Cuantiles teóricos (Normal)')
axes[2].set_ylabel('Cuantiles observados')
axes[2].text(0.05, 0.92, f'R² Q-Q = {r_qq**2:.3f}',
             transform=axes[2].transAxes, fontsize=9, color='#333')

plt.tight_layout()
plt.show()

stat_sw, p_sw = stats.shapiro(residuos)
print(f'Test de Shapiro-Wilk: W = {stat_sw:.4f}, p = {p_sw:.4f}')
print(f'→ {"Los residuos NO siguen una distribución normal (p < 0.05)." if p_sw < 0.05 else "No se rechaza normalidad (p ≥ 0.05)."}')

---
## 10. Interpretación y conclusiones

In [ ]:
resumen = pd.DataFrame({
    'Parámetro'  : ['Backend de cómputo', 'Dispositivo', 'β₀ (Intercepto)',
                    'β₁ (city-mpg)', 'RMSE', 'MAE', 'R² Prueba', 'CV-R²'],
    'Valor'      : [str(model), str(DEVICE), beta0, beta1, rmse, mae, r2, cv_r2],
    'Unidad'     : ['—', '—', 'USD', 'USD/mpg', 'USD', 'USD', '—', '—']
})

display(
    resumen.style
    .format({'Valor': lambda v: f'{float(v):,.2f}'
             if isinstance(v, (int, float)) else v})
    .hide(axis='index')
    .set_caption('Resumen del modelo — Regresión Lineal Simple')
)

### Hallazgos principales

| Aspecto | Hallazgo |
|---|---|
| **Relación** | Correlación negativa entre `city-mpg` y `price`: los autos más eficientes tienden a ser más baratos. |
| **Pendiente** | Cada mpg adicional reduce el precio predicho en ≈ $742 USD. |
| **R² = 0.39** | El modelo explica solo el 39% de la varianza del precio; el 61% restante depende de otras variables no incluidas. |
| **RMSE ≈ $8,615** | El error típico representa ~72% del precio mediano, haciéndolo poco confiable para estimaciones individuales. |
| **Residuos** | El Q-Q plot y Shapiro-Wilk revelan residuos **no normales**, indicando heterocedasticidad y sesgo en los extremos del rango de precios. |
| **Soporte CUDA** | `LinearRegressionDevice` selecciona automáticamente cuML, PyTorch-CUDA o sklearn-CPU, sin cambiar ninguna línea del análisis. El speedup GPU escala con el volumen de datos. |

### Recomendaciones

1. **Regresión múltiple** — Incorporar `engine-size`, `horsepower`, `make` y `body-style`.
2. **Transformación logarítmica** — Aplicar `log(price)` como objetivo para corregir la asimetría.
3. **Detección de outliers** — Tratar valores atípicos que inflan el RMSE.
4. **Modelos no lineales** — Explorar polinomios de grado 2 o árboles de regresión.
5. **Escalado a GPU** — Para datasets grandes (> 1 M filas), `cuml.LinearRegression` ofrece speedups de ×10–×50 respecto a sklearn en CPU.